<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Structured_Outputs_Project_Log_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: Log Analysis and Structured Incident Reports

Companion notebook for the lesson **Applied Structured Outputs: Three Mini Projects**.

We turn raw application logs into a typed incident report an alerting stack can consume. Logs bring two problems that plain ticket text did not have: they contain secrets, and most of their bulk is noise. So the pipeline has four stages: **redact**, **aggregate**, **report**, **enforce**.

**What you will build:**

1. A redaction function that strips secrets before any text reaches an external API, and counts what it strips.
2. A plain-Python aggregation step that hands the model computed statistics instead of asking it to count.
3. A structured `IncidentReport` generated by the model and re-validated with Pydantic.
4. A deterministic security override that escalates a leaked credential no matter what the report says.
5. A confidence gate that routes uncertain reports to a human before anyone gets paged.

## Install Packages and Set Up the Provider

Pick your provider by setting `PROVIDER` below. Gemini is the course default and its free tier covers this notebook.

> **Colab users:** store your API key in **Secrets** (the key icon in the left sidebar) under the name shown for your provider (`GOOGLE_API_KEY`, `OPENAI_API_KEY`, or `ANTHROPIC_API_KEY`). The cell falls back to an interactive prompt if no secret is found.

In [1]:
# Shared install profile for the structured outputs project notebooks (pin set checked August 2026)
!pip install -q google-genai==2.18.0 openai==3.0.0 anthropic==0.122.0 pydantic==2.13.4 pandas==3.0.5 tqdm==4.70.0

In [2]:
import os
import getpass

PROVIDER = "gemini"  # "gemini" | "openai" | "anthropic"

KEY_ENV = {
    "gemini": "GOOGLE_API_KEY",
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}
env_var = KEY_ENV[PROVIDER]

# Option 1: Colab Secrets (recommended)
try:
    from google.colab import userdata

    os.environ[env_var] = userdata.get(env_var)
except Exception:
    pass

# Option 2: interactive prompt (fallback)
if not os.getenv(env_var):
    os.environ[env_var] = getpass.getpass(f"Enter {env_var}: ")

print(f"[OK] {env_var} is set")

[OK] GOOGLE_API_KEY is set


### The `extract()` Helper

All project code goes through one function, `extract(prompt, schema)`. It sends a prompt and returns a validated Pydantic object, using the native structured output API of whichever provider you selected. We use each provider's small, fast model: extraction is high-volume, low-difficulty work, and the flagship models cost several times more per token while adding little on tasks this constrained. (Model IDs current as of August 2026. Swap in the provider's latest small model if these have been superseded.)

In [3]:
from pydantic import BaseModel

MODELS = {
    "gemini": "gemini-3.5-flash-lite",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-haiku-4-5",
}

if PROVIDER == "gemini":
    from google import genai

    client = genai.Client()
elif PROVIDER == "openai":
    from openai import OpenAI

    client = OpenAI()
elif PROVIDER == "anthropic":
    import anthropic

    client = anthropic.Anthropic()


def extract(
    prompt: str,
    schema: type[BaseModel],
    system: str | None = None,
    model: str | None = None,
) -> BaseModel:
    """Send a prompt and return a validated instance of `schema`."""
    if PROVIDER == "gemini":
        response = client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config={
                "system_instruction": system,
                "response_mime_type": "application/json",
                "response_schema": schema,
            },
        )
        if response.parsed is None:
            raise ValueError("Model returned no parseable output")
        return response.parsed
    if PROVIDER == "openai":
        response = client.responses.parse(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            text_format=schema,
            reasoning={"effort": "none"},
        )
        return response.output_parsed
    if PROVIDER == "anthropic":
        response = client.messages.parse(
            model=model or MODELS["anthropic"],
            max_tokens=2048,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
            output_format=schema,
        )
        return response.parsed_output
    raise ValueError(f"Unknown provider: {PROVIDER}")


print(f"[OK] extract() ready, provider={PROVIDER}, model={MODELS[PROVIDER]}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[OK] extract() ready, provider=gemini, model=gemini-3.5-flash-lite


## The Synthetic Log Dataset

This excerpt simulates an API gateway during a bad deploy. It includes normal requests, a burst of 500 errors, a database timeout, and one leaked secret. A user pasted an API key into the course-search box, so the key sits in the logged query string of an ordinary `INFO` request. Leaks do not announce their severity.

In [4]:
raw_logs = """
2026-07-10T12:00:01Z INFO  api-gw request_id=aa1 path=/courses/search status=200 latency_ms=120
2026-07-10T12:00:05Z ERROR api-gw request_id=aa2 path=/courses/search status=500 latency_ms=5100 err="ValueError: invalid param: dateRange"
2026-07-10T12:00:06Z ERROR api-gw request_id=aa3 path=/courses/search status=500 latency_ms=4980 err="ValueError: invalid param: dateRange"
2026-07-10T12:00:07Z INFO  api-gw request_id=aa4 path=/courses/search q=sk_live_ABC123XYZ status=200 latency_ms=95
2026-07-10T12:00:08Z ERROR db  query="SELECT ..." status=timeout duration_ms=30000
2026-07-10T12:00:09Z ERROR api-gw request_id=aa5 path=/courses/search status=500 latency_ms=5200 err="ValueError: invalid param: dateRange"
2026-07-10T12:02:10Z INFO  api-gw request_id=aa6 path=/home status=200 latency_ms=80
""".strip()

print(raw_logs.splitlines()[0])
print("lines:", len(raw_logs.splitlines()))

2026-07-10T12:00:01Z INFO  api-gw request_id=aa1 path=/courses/search status=200 latency_ms=120
lines: 7


## Step 1: Redact Secrets Before the LLM

Logs routinely capture API keys, session tokens, and personal data. Sending them to an external API copies them into another company's infrastructure, expands your audit surface, and can violate data-minimization requirements under frameworks like GDPR and SOC 2.

We use `re.subn` rather than `re.sub`: it returns the new text along with how many substitutions it made, so the redaction step also counts the secrets it found. That count becomes load-bearing in Step 4.

The verification step matters as much as the regex: a redaction function that silently stops matching is worse than none, because you believe you are protected. Keep assertions like these in the pipeline itself.

In [5]:
import re


def redact(text: str) -> tuple[str, int]:
    """Replace sensitive values with placeholders before LLM processing."""
    text, n_keys = re.subn(r"sk_live_[A-Za-z0-9]+", "sk_live_REDACTED", text)
    text, n_tokens = re.subn(r"(token=)(\S+)", r"\1REDACTED", text)
    return text, n_keys + n_tokens


safe_logs, secrets_found = redact(raw_logs)
print("Contains raw key: ", "sk_live_ABC" in safe_logs)
print("Contains REDACTED:", "REDACTED" in safe_logs)
print("Secrets redacted: ", secrets_found)
assert "sk_live_ABC" not in safe_logs

Contains raw key:  False
Contains REDACTED: True
Secrets redacted:  1


> **Security note:** a regex denylist only catches patterns you anticipated. For higher-stakes pipelines, invert the approach: parse the logs into fields and pass an explicit allowlist of safe fields to the model, dropping everything else by default. The exercises walk you through both upgrades.

## Step 2: Aggregate Statistics in Plain Python

We could dump the redacted logs into the prompt and ask for a report. That works at seven lines and degrades at seven million. Counting error rates and grouping failure signatures is deterministic work that Python does exactly and for free, so we do it before the model call and hand the model the totals. The model then interprets numbers instead of counting them, which LLMs do unreliably.

In [6]:
from collections import Counter


def aggregate_logs(log_text: str) -> dict:
    lines = log_text.splitlines()
    status_5xx, err_counter, path_5xx = 0, Counter(), Counter()

    for line in lines:
        status = re.search(r"status=(\d+)", line)
        path = re.search(r"path=(\S+)", line)
        err = re.search(r'err="([^"]+)"', line)
        if status and int(status.group(1)) >= 500:
            status_5xx += 1
            path_5xx[path.group(1) if path else "unknown"] += 1
            if err:
                err_counter[err.group(1)] += 1

    return {
        "total_lines": len(lines),
        "errors_5xx": status_5xx,
        "top_error_signatures": err_counter.most_common(5),
        "top_error_paths": path_5xx.most_common(5),
    }


stats = aggregate_logs(safe_logs)
stats

{'total_lines': 7,
 'errors_5xx': 3,
 'top_error_signatures': [('ValueError: invalid param: dateRange', 3)],
 'top_error_paths': [('/courses/search', 3)]}

## Step 3: Generate the Incident Report

The report schema mirrors what an SRE would write by hand: impact, suspected cause, evidence, recommended actions, plus an escalation flag your paging system can act on.

In [7]:
from pydantic import Field


class IncidentReport(BaseModel):
    incident_title: str = Field(description="Short title")
    impact: str = Field(description="Who or what is affected, and how badly")
    suspected_root_cause: str = Field(description="Best-guess root cause")
    evidence: list[str] = Field(description="Evidence drawn from the logs and stats")
    recommended_actions: list[str] = Field(
        description="Concrete next steps for engineers"
    )
    needs_escalation: bool = Field(description="True if on-call escalation is needed")
    confidence: float = Field(ge=0, le=1, description="Self-reported confidence, 0-1")


prompt = f"""You are an SRE assistant. Given the log excerpt and precomputed stats,
write an incident report for engineers.

LOGS:
{safe_logs}

STATS:
{stats}
"""

report = extract(prompt, IncidentReport)
print("[OK] report generated")

[OK] report generated


In [8]:
import json

print(json.dumps(report.model_dump(), indent=2))

{
  "incident_title": "API Gateway 500 Errors on Course Search Due to Invalid DateRange Parameter",
  "impact": "Users attempting to search courses via /courses/search are experiencing HTTP 500 errors and high latency due to invalid dateRange parameters.",
  "suspected_root_cause": "The application code fails to properly sanitize or handle the 'dateRange' parameter on the /courses/search endpoint, triggering a ValueError and subsequent high latency/timeouts.",
  "evidence": [
    "3 errors of type ValueError: invalid param: dateRange observed on /courses/search",
    "High latency ranging from 4980ms to 5200ms recorded alongside the 500 status codes",
    "A database timeout of 30000ms was observed during the incident window"
  ],
  "recommended_actions": [
    "Investigate the input validation logic for the 'dateRange' parameter on the /courses/search endpoint",
    "Implement proper error handling to return HTTP 400 instead of HTTP 500 for malformed parameters",
    "Review database 

Look at the evidence list: the leaked key is missing. The model wrote a solid report about the 500 errors and passed over `q=sk_live_REDACTED` without comment. Nothing in the pipeline pushed it to do otherwise. The aggregation counts only 5xx errors, so the key appears nowhere in the stats, and it sits on a routine `INFO` line that no severity filter would flag. On real log volumes, where the prompt carries the stats plus a sample of lines, the leaking line can miss the sample entirely. On another run, the model may notice the key and flag it. A mention that depends on the model noticing something is judgment, and judgment can miss.

## Step 4: The Security Override

Anything you must catch every single time, like a leaked credential, belongs in deterministic code. Step 1 already computed the fact: `redact()` counted its substitutions. The override pattern from Project 1 turns that count into a guarantee.

In [9]:
def apply_security_overrides(report: IncidentReport, secrets_found: int) -> IncidentReport:
    if secrets_found > 0:
        report.needs_escalation = True
        report.recommended_actions.append(
            "Rotate the leaked credential and fix log sanitization"
        )
    return report


report = apply_security_overrides(report, secrets_found)
print("needs_escalation:", report.needs_escalation)
print("appended action: ", report.recommended_actions[-1])

needs_escalation: True
appended action:  Rotate the leaked credential and fix log sanitization


The rotation action now reaches the on-call engineer whether or not the model noticed the leak. The trade-off is redundancy: in runs where the model does flag the key, the appended action repeats it. A duplicate recommendation costs nothing next to a missed leak. In production you would also add `secrets_found` to the stats dict, so the report cites the leak as a computed fact instead of relying on the model to spot it.

## Step 5: The Confidence Gate

Same pattern as Project 1: the pipeline acts on confident reports and queues uncertain ones for a person. Here the action is paging someone at 3am, so the gate earns its keep.

In [10]:
CONFIDENCE_THRESHOLD = 0.7


def route_report(report: IncidentReport) -> str:
    if report.confidence < CONFIDENCE_THRESHOLD:
        return "human-review-queue"
    if report.needs_escalation:
        return "page-on-call"
    return "log-only"


print("Routing decision:", route_report(report))

Routing decision: page-on-call


## Exercises

1. **More redaction patterns.** Extend `redact()` to cover email addresses, IPv4 addresses, and `Bearer` tokens. Write the assertions first, then the regexes.
2. **The allowlist version.** Parse each log line into fields and rebuild the pipeline so only an explicit allowlist of fields ever reaches the prompt.
3. **Chunk and merge.** Split a larger log file into chunks, generate one report per chunk, then merge them: union the evidence and actions, OR the escalation flags, and average confidence. When does merging produce a worse report than one big prompt?
4. **Wire the gate.** Collect reports routed to `human-review-queue` in a list and render them as a review table with `pandas`.